In [0]:

"""
Project      : Credit Card Fraud Analytics Pipeline
Notebook     : 04_Data_Analysis.py
Author       : Atchaya P
Environment  : Databricks Serverless
Description  : Business analytics using PySpark aggregations, window functions and Spark SQL.
"""

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# -------------------------------------------------------------------
# Read Engineered Dataset
# -------------------------------------------------------------------

DATA_PATH = "/Workspace/Users/atchaya.analyst@gmail.com/Fraud Analytics/fraudTrain.csv"

transactions_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(DATA_PATH)
)

# -------------------------------------------------------------------
# Basic Feature Engineering (if Notebook 03 not executed)
# -------------------------------------------------------------------

transactions_df = (
    transactions_df
    .withColumn("Customer_Name", concat_ws(" ", col("first"), col("last")))
    .withColumn("Transaction_Date", to_date("trans_date_trans_time"))
    .withColumn("Month_Name", date_format("Transaction_Date","MMMM"))
    .withColumn("Weekend_Flag",
                when(dayofweek("Transaction_Date").isin(1,7),"Weekend")
                .otherwise("Weekday"))
)

print("="*80)
print("Credit Card Fraud Analytics")
print("="*80)

# -------------------------------------------------------------------
# KPI Summary
# -------------------------------------------------------------------

kpi_df = transactions_df.agg(
    count("*").alias("Total_Transactions"),
    round(sum("amt"),2).alias("Total_Revenue"),
    round(avg("amt"),2).alias("Average_Transaction"),
    sum("is_fraud").alias("Fraud_Transactions")
)

display(kpi_df)

# -------------------------------------------------------------------
# Revenue by Category
# -------------------------------------------------------------------

category_df = (
    transactions_df
    .groupBy("category")
    .agg(
        round(sum("amt"),2).alias("Revenue"),
        count("*").alias("Transactions"),
        round(avg("amt"),2).alias("Average_Amount")
    )
    .orderBy(desc("Revenue"))
)

display(category_df)
# Visualization: Bar Chart

# -------------------------------------------------------------------
# Fraud by Category
# -------------------------------------------------------------------

fraud_category = (
    transactions_df
    .groupBy("category")
    .agg(sum("is_fraud").alias("Fraud_Count"))
    .orderBy(desc("Fraud_Count"))
)

display(fraud_category)
# Visualization: Horizontal Bar

# -------------------------------------------------------------------
# Top Merchants
# -------------------------------------------------------------------

merchant_df = (
    transactions_df
    .groupBy("merchant")
    .agg(round(sum("amt"),2).alias("Revenue"))
    .orderBy(desc("Revenue"))
    .limit(10)
)

display(merchant_df)
# Visualization: Horizontal Bar

# -------------------------------------------------------------------
# Revenue by State
# -------------------------------------------------------------------

state_df = (
    transactions_df
    .groupBy("state")
    .agg(round(sum("amt"),2).alias("Revenue"))
    .orderBy(desc("Revenue"))
)

display(state_df)
# Visualization: Map or Bar

# -------------------------------------------------------------------
# Top Customers
# -------------------------------------------------------------------

customer_df = (
    transactions_df
    .groupBy("cc_num","Customer_Name")
    .agg(round(sum("amt"),2).alias("Total_Spend"))
)

window_spec = Window.orderBy(desc("Total_Spend"))

customer_df = customer_df.withColumn(
    "Rank",
    dense_rank().over(window_spec)
)

display(customer_df.filter(col("Rank") <= 10))
# Visualization: Bar

# -------------------------------------------------------------------
# Highest Transaction Per Category
# -------------------------------------------------------------------

category_window = Window.partitionBy("category").orderBy(desc("amt"))

highest_txn = (
    transactions_df
    .withColumn("Row_Num", row_number().over(category_window))
    .filter(col("Row_Num") == 1)
)

display(highest_txn)

# -------------------------------------------------------------------
# Running Revenue
# -------------------------------------------------------------------

running_window = (
    Window.orderBy("Transaction_Date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

running_df = transactions_df.withColumn(
    "Running_Revenue",
    sum("amt").over(running_window)
)

display(running_df.select("Transaction_Date","amt","Running_Revenue"))

# -------------------------------------------------------------------
# Fraud Percentage by Category
# -------------------------------------------------------------------

fraud_percent = (
    transactions_df
    .groupBy("category")
    .agg(
        count("*").alias("Total"),
        sum("is_fraud").alias("Fraud")
    )
    .withColumn(
        "Fraud_Percentage",
        round(col("Fraud")*100/col("Total"),2)
    )
    .orderBy(desc("Fraud_Percentage"))
)

display(fraud_percent)
# Visualization: Bar

# -------------------------------------------------------------------
# Spark SQL
# -------------------------------------------------------------------

transactions_df.createOrReplaceTempView("fraud_transactions")

display(spark.sql("""
SELECT
    category,
    COUNT(*) AS Transactions,
    ROUND(SUM(amt),2) AS Revenue,
    ROUND(AVG(amt),2) AS Average_Amount
FROM fraud_transactions
GROUP BY category
ORDER BY Revenue DESC
"""))

display(spark.sql("""
SELECT
    state,
    SUM(is_fraud) AS Fraud_Transactions
FROM fraud_transactions
GROUP BY state
ORDER BY Fraud_Transactions DESC
LIMIT 10
"""))

display(spark.sql("""
SELECT
    Month_Name,
    ROUND(SUM(amt),2) AS Revenue
FROM fraud_transactions
GROUP BY Month_Name
ORDER BY Revenue DESC
"""))

display(spark.sql("""
SELECT
    Weekend_Flag,
    COUNT(*) AS Transactions,
    SUM(is_fraud) AS Fraud_Transactions
FROM fraud_transactions
GROUP BY Weekend_Flag
"""))

print("="*80)
print("Notebook 04 Completed Successfully")
print("="*80)



Credit Card Fraud Analytics


Total_Transactions,Total_Revenue,Average_Transaction,Fraud_Transactions
1296675,9.12224289E7,70.35,7506


category,Revenue,Transactions,Average_Amount
grocery_pos,1.446082238E7,123638,116.96
shopping_pos,9307993.61,116672,79.78
shopping_net,8625149.68,97543,88.42
gas_transport,8351732.29,131659,63.43
home,7173928.11,123115,58.27
kids_pets,6503680.16,113035,57.54
entertainment,6036678.56,94014,64.21
misc_net,5117709.26,63287,80.87
misc_pos,5009582.5,79655,62.89
food_dining,4672459.44,91461,51.09


category,Fraud_Count
grocery_pos,1743
shopping_net,1713
misc_net,915
shopping_pos,843
gas_transport,618
misc_pos,250
kids_pets,239
entertainment,233
personal_care,220
home,198


merchant,Revenue
fraud_Kilback LLC,391078.15
fraud_Bradtke PLC,302481.25
fraud_Doyle Ltd,300971.37
fraud_Hackett-Lueilwitz,300208.14
"fraud_Schumm, Bauch and Ondricka",299115.14
fraud_Rau and Sons,298354.77
fraud_Goodwin-Nitzsche,298083.31
fraud_Pacocha-O'Reilly,297584.38
fraud_Murray-Smitham,296982.73
fraud_Bauch-Raynor,295721.2


...

category,Transactions,Revenue,Average_Amount
grocery_pos,123638,1.446082238E7,116.96
shopping_pos,116672,9307993.61,79.78
shopping_net,97543,8625149.68,88.42
gas_transport,131659,8351732.29,63.43
home,123115,7173928.11,58.27
kids_pets,113035,6503680.16,57.54
entertainment,94014,6036678.56,64.21
misc_net,63287,5117709.26,80.87
misc_pos,79655,5009582.5,62.89
food_dining,91461,4672459.44,51.09


state,Fraud_Transactions
NY,555
TX,479
PA,458
CA,326
OH,321
FL,281
IL,248
MI,238
AL,215
MN,207


Month_Name,Revenue
May,1.030494014E7
March,1.019000248E7
June,1.014338217E7
December,9918179.18
April,9452834.36
January,7422814.52
February,6974606.28
August,6047288.65
July,6044026.74
September,4949834.42


Weekend_Flag,Transactions,Fraud_Transactions
Weekday,845139,5063
Weekend,451536,2443


Notebook 04 Completed Successfully
